In [14]:
import os
import sys
import findspark

# 1. On stabilise les chemins
spark_home = r"C:\tools\spark-3.5.8-bin-hadoop3"
os.environ['SPARK_HOME'] = spark_home
os.environ['HADOOP_HOME'] = r"C:\hadoop"
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_LOCAL_IP'] = "127.0.0.1" # Force la connexion locale

# 2. On initialise findspark
findspark.init(spark_home)

# 3. On importe PySpark
from pyspark.sql import SparkSession
import pyspark

In [15]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [17]:
df = spark.read.option("header","true").parquet("fhv_tripdata_2021-02.parquet")

In [18]:
df.show()

+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|dispatching_base_num|    pickup_datetime|   dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|
+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|              B00013|2021-02-01 00:01:00|2021-02-01 01:33:00|        NULL|        NULL|   NULL|                B00014|
|     B00021         |2021-02-01 00:55:40|2021-02-01 01:06:20|       173.0|        82.0|   NULL|       B00021         |
|     B00021         |2021-02-01 00:14:03|2021-02-01 00:28:37|       173.0|        56.0|   NULL|       B00021         |
|     B00021         |2021-02-01 00:27:48|2021-02-01 00:35:45|        82.0|       129.0|   NULL|       B00021         |
|              B00037|2021-02-01 00:12:50|2021-02-01 00:26:38|        NULL|       225.0|   NULL|                B00037|
|              B00037|2021-02-01 00:00:3

In [42]:
df.printSchema

<bound method DataFrame.printSchema of DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: double, DOLocationID: double, SR_Flag: string]>

In [20]:
df.dtypes

[('dispatching_base_num', 'string'),
 ('pickup_datetime', 'timestamp_ntz'),
 ('dropOff_datetime', 'timestamp_ntz'),
 ('PUlocationID', 'double'),
 ('DOlocationID', 'double'),
 ('SR_Flag', 'int'),
 ('Affiliated_base_number', 'string')]

In [35]:
from pyspark.sql import types
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.DoubleType(), True),
    types.StructField('DOLocationID', types.DoubleType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

In [36]:
df = spark.read.option("header","true").schema(schema).parquet("fhv_tripdata_2021-02.parquet")

In [37]:
df.dtypes

[('hvfhs_license_num', 'string'),
 ('dispatching_base_num', 'string'),
 ('pickup_datetime', 'timestamp'),
 ('dropoff_datetime', 'timestamp'),
 ('PULocationID', 'double'),
 ('DOLocationID', 'double'),
 ('SR_Flag', 'string')]

In [38]:
df = df.repartition(24)

In [39]:
df.write.parquet('fhv_tripdata/2021/02')

In [40]:
df = spark.read.parquet("fhv_tripdata/2021/02")

In [41]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|             NULL|              B02735|2021-02-11 12:36:24|2021-02-11 12:53:17|        NULL|       213.0|   NULL|
|             NULL|              B02989|2021-02-26 15:16:48|2021-02-26 16:20:47|        NULL|        NULL|   NULL|
|             NULL|              B02881|2021-02-25 14:07:32|2021-02-25 15:04:17|        29.0|       130.0|   NULL|
|             NULL|              B00254|2021-02-06 09:56:36|2021-02-06 10:21:49|       262.0|        26.0|   NULL|
|             NULL|              B00095|2021-02-18 07:16:20|2021-02-18 07:19:36|        NULL|       198.0|   NULL|
|             NULL|              B02573|2021-02-23 11:28:22|2021-02-23 11:35:59|

In [50]:
df.select("pickup_datetime","dropoff_datetime","PULocationID","DOLocationID") \
.filter(df.dispatching_base_num == 'B00457') \
.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-02-17 06:53:55|2021-02-17 07:05:21|        NULL|       243.0|
|2021-02-03 08:59:03|2021-02-03 09:19:31|        NULL|       174.0|
|2021-02-02 13:24:49|2021-02-02 13:39:18|        NULL|       254.0|
|2021-02-18 09:10:12|2021-02-18 09:33:44|        NULL|        69.0|
|2021-02-08 12:43:18|2021-02-08 13:03:39|        NULL|       126.0|
|2021-02-05 16:20:44|2021-02-05 16:24:34|        NULL|       242.0|
|2021-02-16 08:02:00|2021-02-16 08:03:36|        NULL|        78.0|
|2021-02-27 08:53:46|2021-02-27 09:02:06|        NULL|        69.0|
|2021-02-05 00:55:17|2021-02-05 01:02:59|        NULL|        31.0|
|2021-02-27 02:26:02|2021-02-27 02:27:38|        NULL|       241.0|
|2021-02-19 16:42:21|2021-02-19 16:58:48|        NULL|       243.0|
|2021-02-12 18:06:46|2021-02-12 18:23:08|       

In [49]:
df.groupBy("hvfhs_license_num").count().show()

+-----------------+-------+
|hvfhs_license_num|  count|
+-----------------+-------+
|             NULL|1037692|
+-----------------+-------+



In [51]:
from pyspark.sql import functions as F

In [54]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
.withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
.select('pickup_date','dropoff_date') \
.show()


+-----------+------------+
|pickup_date|dropoff_date|
+-----------+------------+
| 2021-02-11|  2021-02-11|
| 2021-02-26|  2021-02-26|
| 2021-02-25|  2021-02-25|
| 2021-02-06|  2021-02-06|
| 2021-02-18|  2021-02-18|
| 2021-02-23|  2021-02-23|
| 2021-02-16|  2021-02-16|
| 2021-02-23|  2021-02-23|
| 2021-02-28|  2021-02-28|
| 2021-02-19|  2021-02-19|
| 2021-02-14|  2021-02-14|
| 2021-02-17|  2021-02-17|
| 2021-02-05|  2021-02-06|
| 2021-02-02|  2021-02-02|
| 2021-02-03|  2021-02-03|
| 2021-02-11|  2021-02-11|
| 2021-02-27|  2021-02-27|
| 2021-02-05|  2021-02-05|
| 2021-02-17|  2021-02-17|
| 2021-02-24|  2021-02-24|
+-----------+------------+
only showing top 20 rows

